# 2주차 과제 베이스라인 — 감귤 착과량 예측

감귤나무의 생육 정보로 착과량(열매가 열린 양)을 예측합니다. 데이터는 README에 안내된 [감귤 착과량 예측 AI 경진대회](https://dacon.io/competitions/official/236038) 페이지에서 내려받을 수 있습니다. 기본 시도에는 `train.csv`만 필요하며, `test.csv`는 선택 탐색에서 최종 예측을 만들 때 사용합니다. 파일은 `dataset/open/extracted/감귤 착과량 예측 AI 경진대회/`에 둡니다. 이 베이스라인은 **식별자 1개(`ID`), 입력 4개, 목표값 1개를 포함한 train 6개 열**을 불러옵니다.

원본 데이터에는 2022년 9월 1일부터 11월 28일까지 나무별 일별 `새순`·`엽록소` 측정치 178개 컬럼이 더 있습니다. 이번 과제의 기본 범위에서는 이 컬럼을 제외하고 시작합니다. 어떤 변수를 사용할지 판단하는 과정도 중요한 학습 내용이며, 기본 분석을 마친 뒤 이 컬럼을 추가해 검증 성능이 달라지는지 비교할 수 있습니다.

이번 과제에서는 낮은 RMSE를 만드는 것보다 **직접 시도한 코드, 만난 오류, 문제를 더 작게 나눈 과정, 다음 행동**을 중요하게 기록합니다.

- ✅ **기본 시도**: 변수 하나로 학습·검증 분할, 회귀 학습, RMSE 확인까지 실행합니다.
- 🧩 **문제 분해**: 변수 중복, 잔차, 파일·코드 오류 중 하나를 골라 확인합니다.
- 🌱 **선택 탐색**: 다중회귀, 추가 변수, 최종 예측을 이어서 살펴봅니다.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

basic_cols = ['ID', '수고(m)', '수관폭1(min)', '수관폭2(max)', '수관폭평균']
start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [start, *start.parents] if (p / '02주차' / 'assignment_baseline.ipynb').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('README.md와 02주차 폴더가 있는 프로젝트 안에서 노트북을 실행합니다.')

data_dir = PROJECT_ROOT / 'dataset' / 'open' / 'extracted' / '감귤 착과량 예측 AI 경진대회'
train_path = data_dir / 'train.csv'
test_path = data_dir / 'test.csv'
train_available = train_path.is_file()
test_available = test_path.is_file()
print('확인한 데이터 폴더:', data_dir)
print('기본 필수 train.csv:', '있음' if train_available else '없음')
print('선택 탐색 test.csv:', '있음' if test_available else '없음 — 기본 시도에는 필요하지 않습니다.')

if train_available:
    DATA_MODE = '실제 데이터'
    train = pd.read_csv(train_path, usecols=basic_cols + ['착과량(int)'])
    test = pd.read_csv(test_path, usecols=basic_cols) if test_available else None
else:
    DATA_MODE = '연습용 소규모 예시 데이터'
    train = pd.DataFrame({
        'ID': [f'DEMO_{i:02d}' for i in range(12)],
        '수고(m)': [210, 225, 230, 240, 245, 255, 260, 270, 280, 285, 295, 305],
        '수관폭1(min)': [220, 235, 238, 250, 255, 265, 270, 282, 290, 300, 310, 318],
        '수관폭2(max)': [230, 245, 250, 260, 267, 275, 282, 294, 302, 312, 320, 332],
        '착과량(int)': [95, 110, 108, 125, 132, 140, 151, 158, 168, 172, 185, 194],
    })
    train['수관폭평균'] = (train['수관폭1(min)'] + train['수관폭2(max)']) / 2
    test = train.drop(columns='착과량(int)').iloc[:3].copy()
    print('실제 파일을 준비하기 전까지 예시 데이터로 실행 흐름만 연습합니다.')

print('데이터 모드:', DATA_MODE)
print('train 크기:', train.shape)
print('test 크기:', test.shape if test is not None else '파일 없음 — 선택 탐색을 건너뜁니다.')


In [ ]:
train.head()


## 이제 분석을 이어서 작성합니다

`example.ipynb`의 따릉이 대여량 예측과 같은 순서로 진행합니다. 먼저 ✅ 표시가 있는 두 단계를 실행합니다. 나머지 셀은 실행 가능한 첫 코드이지만, 기본 기록을 남긴 뒤 천천히 살펴봐도 됩니다.


### 1. ✅ 기본 시도 — 내부 학습·검증 데이터 분리

변수를 고르기 전에 competition `train`을 내부 학습 데이터와 검증 데이터로 나눕니다. 이후 산점도·상관관계와 변수 선택은 내부 학습 데이터에서만 수행하고, 검증 데이터는 선택한 모델을 확인할 때 사용합니다.


In [ ]:
fit_df, valid_df = train_test_split(
    train, test_size=0.2, random_state=42, shuffle=True
)
print('학습/검증 행 수:', len(fit_df), len(valid_df))
print('행 겹침:', len(set(fit_df.index) & set(valid_df.index)))


### 2. ✅ 기본 시도 — 변수 선택과 단순선형회귀

`fit_df`에서 `수고`, `수관폭1`, `수관폭2`, `수관폭평균`과 `착과량(int)`의 관계를 먼저 살펴봅니다. 관계가 뚜렷해 보이는 변수 하나로 단순회귀를 학습한 뒤 `valid_df`로 RMSE를 계산합니다. 서로 비슷한 정보를 담은 변수를 함께 사용할 때는 중복 정보가 계수 해석에 미치는 영향도 확인합니다.


In [ ]:
feature_cols = ['수고(m)', '수관폭1(min)', '수관폭2(max)', '수관폭평균']
print(fit_df[feature_cols + ['착과량(int)']].corr()['착과량(int)'].sort_values())

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(fit_df['수고(m)'], fit_df['착과량(int)'], alpha=0.6)
ax.set_xlabel('수고(m)')
ax.set_ylabel('착과량(int)')
ax.set_title('학습 데이터: 수고와 착과량')
plt.show()

simple_features = ['수고(m)']
simple_model = LinearRegression().fit(fit_df[simple_features], fit_df['착과량(int)'])
pred_simple = simple_model.predict(valid_df[simple_features])
rmse_simple = mean_squared_error(valid_df['착과량(int)'], pred_simple) ** 0.5
print('단순회귀 검증 RMSE:', round(rmse_simple, 3))


### 3. 🌱 선택 탐색 — 다중선형회귀

여러 변수를 함께 사용한 다중회귀 모델을 만들고, 같은 검증 데이터에서 단순회귀와 RMSE를 비교합니다.


In [ ]:
# 수관폭평균은 두 원측정치의 정확한 평균이므로 함께 넣지 않습니다.
multi_features = ['수고(m)', '수관폭1(min)', '수관폭2(max)']
multi_model = LinearRegression().fit(fit_df[multi_features], fit_df['착과량(int)'])
pred_multi = multi_model.predict(valid_df[multi_features])
rmse_multi = mean_squared_error(valid_df['착과량(int)'], pred_multi) ** 0.5
print('단순회귀 검증 RMSE:', round(rmse_simple, 3))
print('다중회귀 검증 RMSE:', round(rmse_multi, 3))


### 4. 🌱 선택 탐색 — 계수 해석

각 변수의 계수를 다른 입력을 고정했을 때의 조건부 차이로 해석합니다. 예를 들어 '다른 변수가 같을 때 수고가 1m 늘어나면 모델의 착과량 예측값은 평균적으로 얼마나 달라집니까?'라고 질문할 수 있습니다.


In [ ]:
coefficients = pd.Series(multi_model.coef_, index=multi_features, name='계수')
display(coefficients.to_frame())
print('힌트: 다른 입력을 고정했을 때 해당 변수가 1만큼 달라질 때의 예측값 차이로 읽습니다.')


### 5. 🧩 문제 분해 — 잔차 분석

기본 시도에서 만든 단순회귀 예측으로 실제값에서 예측값을 뺀 잔차를 확인합니다. 앞의 다중회귀 선택 셀을 실행하지 않아도 됩니다. 곡선, 부채꼴, 극단값처럼 모델이 충분히 설명하지 못한 패턴이 남아 있는지 한 가지만 살펴봅니다.


In [ ]:
residuals = valid_df['착과량(int)'].to_numpy() - pred_simple
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(pred_simple, residuals)
axes[0].axhline(0, color='red')
axes[0].set_xlabel('예측값')
axes[0].set_ylabel('잔차(실제-예측)')
axes[0].set_title('예측값과 잔차')
axes[1].hist(residuals, bins=min(12, max(3, len(residuals))))
axes[1].set_title('잔차 분포')
plt.tight_layout()
plt.show()


### 6. 🌱 선택 탐색 — test.csv로 예측해보기

`test.csv`는 정답이 없는 최종 예측용 데이터입니다. 학습한 모델로 예측값을 만든 뒤 `sample_submission.csv`와 같은 형식으로 저장해 제출 파일의 구조를 확인할 수 있습니다.


In [ ]:
if DATA_MODE != '실제 데이터':
    print('예시 데이터 모드에서는 제출 파일을 만들지 않습니다.')
elif test is None:
    print('test.csv가 없어 선택 제출 예측을 건너뜁니다. 기본 학습 기록은 이미 완료할 수 있습니다.')
else:
    final_model = LinearRegression().fit(train[multi_features], train['착과량(int)'])
    submission = pd.DataFrame({
        'ID': test['ID'],
        '착과량(int)': final_model.predict(test[multi_features]),
    })
    display(submission.head())
    print('저장하기 전 sample_submission.csv의 열 이름과 순서를 비교합니다.')


## 이번 주 학습 기록

실행 성공보다 시도·오류·문제 분해 기록을 우선합니다. 아래 네 줄을 자신의 말로 채웁니다.

- 질문 1개: 예) 수고 하나로 착과량을 어느 정도 예측할 수 있습니까?
- 실행 또는 실행 시도 1개: 실행한 셀과 데이터 모드
- 관찰 결과 또는 오류 1개: RMSE, 그래프 모양, 누락 파일명 또는 오류 메시지
- 다음 행동 1개: 파일 배치, 변수 변경, 잔차 확인 등

작은 힌트: 오류가 나면 `파일 → 열 이름 → X와 y의 행 수 → 결측치 → 모델 입력 모양` 순서로 하나씩 확인합니다.

> **여기까지 하면 이번 주 기록 완료입니다.** 다중회귀와 제출 파일은 선택 탐색이며, 잔차 분석은 기본 결과를 조금 더 살펴보고 싶을 때 진행합니다.
